# 01 — Data Cleaning

## Purpose
Load the raw-but-concatenated data (`datanomodifier.csv`) and apply all **cleaning and
standardization** steps before any analysis begins.

After this notebook you have a clean, consistent dataframe called `df` that is ready
to be analyzed in notebooks 02–05.

## Input
`data/intermediate/datanomodifier.csv` — produced by `00_data_pipeline.ipynb`

## Output
No file is saved here. The cleaned `df` is the result. The analysis notebooks
each reload and re-apply these same cleaning steps.

## Run order
Run after `00_data_pipeline.ipynb`.

In [2]:
import os

# ─── PATH CONFIGURATION ───────────────────────────────────────────────
# Option A — After cloning the repo (default, USE_GITHUB = False)
#   git clone https://github.com/DiegoLarrieta/PanemReto
#   cd PanemReto/notebooks
#   jupyter notebook
#   Paths resolve automatically — no changes needed.
#
# Option B — Read directly from GitHub, e.g. Google Colab (USE_GITHUB = True)
#   Works for notebooks 07 (processed branch CSVs are on GitHub).
#   Notebooks 00-06 need the intermediate files which are NOT in the repo
#   (too large). Run 00_data_pipeline.ipynb locally first to generate them.
# ──────────────────────────────────────────────────────────────────────────

USE_GITHUB   = False
GITHUB_BASE  = "https://media.githubusercontent.com/media/DiegoLarrieta/PanemReto/main"

if USE_GITHUB:
    PROCESSED_DIR    = f"{GITHUB_BASE}/data/processed"
    WEATHER_DIR      = f"{GITHUB_BASE}/data/weather"
    RAW_DIR          = f"{GITHUB_BASE}/data/raw/Complete Data"
    INTERMEDIATE_DIR = None  # not in repo — generate locally with 00_data_pipeline.ipynb
else:
    PROJECT_ROOT     = os.path.abspath(os.path.join(os.getcwd(), ".."))
    PROCESSED_DIR    = os.path.join(PROJECT_ROOT, "data", "processed")
    WEATHER_DIR      = os.path.join(PROJECT_ROOT, "data", "weather")
    RAW_DIR          = os.path.join(PROJECT_ROOT, "data", "raw", "Complete Data")
    INTERMEDIATE_DIR = os.path.join(PROJECT_ROOT, "data", "intermediate")


## Step 1 — Imports and paths

## Step 2 — Load the data

`datanomodifier.csv` is the output of `00_data_pipeline.ipynb`. It already has:
- Modifier rows removed
- `cold_or_warm` column added
- `day_part` column added
- Weather (`tavg`) merged in

We use `low_memory=False` because some columns have mixed types across files.

In [3]:
import pandas as pd


input_path = os.path.join(INTERMEDIATE_DIR, "datanomodifier.csv")
df = pd.read_csv(input_path, low_memory=False)

print(f"Rows: {len(df):,}")
print(f"Columns ({len(df.columns)}): {df.columns.tolist()}")

Rows: 2,579,572
Columns (50): ['sucursal', 'operating_date', 'day_name', 'closing_time', 'captured_time', 'week_number', 'pdv_txn_id', 'order_id', 'order_type', 'order_subtype', 'table_number', 'party_size', 'server', 'terminal', 'capture_terminal', 'action', 'item', 'modifier', 'group_type', 'group', 'description', 'is_modifier', 'quantity', 'unit_price', 'unit_price_with_mods', 'cost_actual', 'cost_with_mods', 'cost_ideal', 'discount', 'subtotal_ticket', 'iva_ticket', 'ieps_ticket', 'total_ticket', 'subtotal_item', 'iva_item', 'ieps_item', 'total_item', 'subtotal_cortesia_cancel', 'iva_cortesia_cancel', 'ieps_cortesia_cancel', 'total_cortesia_cancel', 'subtotal_anulacion', 'iva_anulacion', 'ieps_anulacion', 'total_anulacion', 'clave_platillo', '_source_file', 'tavg', 'cold_or_warm', 'day_part']


## Step 3 — Convert datetime columns

Even though `00_data_pipeline` already parsed these, they get stored as strings in CSV.
We re-parse them here so every notebook has proper `datetime64` types.

- `operating_date` = the business date of the sale (what day the branch was operating)
- `closing_time` = when the ticket was closed at the POS terminal
- `captured_time` = when the individual item was added to the order (most granular timestamp)

In [4]:
datetime_cols = ["operating_date", "closing_time", "captured_time"]
for col in datetime_cols:
    if col in df.columns:
        df[col] = pd.to_datetime(df[col], errors="coerce")

print("Datetime columns after conversion:")
print(df[datetime_cols].dtypes)

Datetime columns after conversion:
operating_date    datetime64[ns]
closing_time      datetime64[ns]
captured_time     datetime64[ns]
dtype: object


## Step 4 — Fix the `is_modifier` column

In the raw POS exports this column contains a mix of: `True`, `False`, `'True'`, `'False'`,
and `NaN`. We standardize it to a proper Python boolean.

Note: at this stage `is_modifier=True` rows should already be removed, but this safeguard
ensures any that slipped through are handled correctly.

In [5]:
df["is_modifier"] = (
    df["is_modifier"]
    .astype("string")
    .str.strip()
    .str.lower()
    .map({"true": True, "false": False})
    .fillna(False)
    .astype(bool)
)

print("is_modifier value counts:")
print(df["is_modifier"].value_counts())

is_modifier value counts:
is_modifier
False    2579572
Name: count, dtype: int64


## Step 5 — Remove beverage groups

**Why remove beverages?**
The two beverage groups (`CAFE Y BEBIDAS CALIENTES` and `JUGOS Y BEBIDAS FRIAS`) dominate
sales volume and would distort the product rankings. More importantly, the business goal
is to forecast **food product demand** for daily stock planning — beverages are either
prepared on-demand or have different supply chain logic.

We completely remove these groups from the dataframe before any analysis.

In [6]:
beverage_groups = ["CAFE Y BEBIDAS CALIENTES", "JUGOS Y BEBIDAS FRIAS"]

rows_before = len(df)
df = df[~df["group"].isin(beverage_groups)].copy()
rows_removed = rows_before - len(df)

print(f"Rows before removing beverages: {rows_before:,}")
print(f"Rows removed (beverages):       {rows_removed:,}")
print(f"Rows remaining:                 {len(df):,}")
print()
print("Remaining product groups:")
print(df["group"].value_counts())

Rows before removing beverages: 2,579,572
Rows removed (beverages):       1,140,501
Rows remaining:                 1,439,071

Remaining product groups:
group
PAN DULCE                         675283
DESAYUNOS                         283169
COMIDAS                           151836
REPOSTERIA                        129362
EXTRAS                             48377
PIZZA                              38674
PRODUCTOS DE TEMPORADA             27744
PAN SALADO                         15384
UBER PAN DULCE                     11621
PANEM MARKETPLACE                  10873
SUBSIDIO                            9134
RAPPI PAN DULCE                     5904
UBER DESAYUNOS                      4511
UBER COMIDAS                        4452
ESPECIALES-                         4387
UBER CAFE Y BEBIDAS CALIENTES       4202
UBER JUGOS Y BEBIDAS FRIAS          2039
UBER REPOSTERIA                     1934
RAPPI DESAYUNOS                     1398
RAPPI COMIDAS                       1304
RAPPI CAFE Y BEBIDAS 

## Step 6 — Standardize branch names

The POS system was reconfigured at some point and some branches appear under two different
names in the data (e.g., 'Panem - Hotel Kavia N' and 'Panem - Hotel Kavia' are the same branch).

We map all legacy names to the canonical current names so every analysis groups them correctly.

In [7]:
sucursal_map = {
    "Panem - Hotel Kavia N"      : "Panem - Hotel Kavia",
    "Panem - Plaza QIN N"        : "Panem - Plaza QIN",
    "Panem - Hospital Zambrano N": "Panem - Hospital Zambrano",
    "Panem - La Carreta N"       : "Panem - Carreta",
}

df["sucursal"] = df["sucursal"].replace(sucursal_map)

print(f"Unique branches: {df['sucursal'].nunique()}")
print(sorted(df["sucursal"].dropna().unique()))

Unique branches: 7
['Panem - Carreta', 'Panem - Credi Club', 'Panem - Hospital Zambrano', 'Panem - Hotel Kavia', 'Panem - Plaza Nativa', 'Panem - Plaza QIN', 'Panem - Punto Valle']


## Step 7 — Fix product name typos

A small number of product names have spelling errors that cause the same product
to appear as two separate items in groupby operations.

We correct known typos here. More may be discovered during EDA and added to this list.

In [8]:
name_fixes = {
    "SANDWITCH ENSALADA POLLO": "SANDWICH ENSALADA POLLO",
}

df["item"] = df["item"].replace(name_fixes)
print(f"Applied {len(name_fixes)} name fix(es)")

Applied 1 name fix(es)


## Step 8 — Data quality check

A quick sanity check before we proceed to analysis: shape, missing values,
and a sample of rows.

In [9]:
print(f"Final shape: {df.shape}")
print()
print("Missing values per column:")
missing = df.isna().sum()
print(missing[missing > 0])

Final shape: (1439071, 50)

Missing values per column:
closing_time                   4717
captured_time                  8778
table_number                 720405
capture_terminal               4078
modifier                    1439071
unit_price                     4078
unit_price_with_mods           4078
cost_actual                    4078
cost_with_mods                 4078
cost_ideal                     4078
discount                       4078
subtotal_ticket                4078
iva_ticket                     4078
ieps_ticket                    4078
total_ticket                   4078
subtotal_item                  6436
iva_item                       6436
ieps_item                      6436
total_item                     6436
subtotal_cortesia_cancel    1418633
iva_cortesia_cancel         1418633
ieps_cortesia_cancel        1418633
total_cortesia_cancel       1418633
subtotal_anulacion          1433126
iva_anulacion               1433126
ieps_anulacion              1433126
total_anu

In [10]:
df.head(10)

,sucursal,operating_date,day_name,closing_time,captured_time,week_number,pdv_txn_id,order_id,order_type,order_subtype,...,total_cortesia_cancel,subtotal_anulacion,iva_anulacion,ieps_anulacion,total_anulacion,clave_platillo,_source_file,tavg,cold_or_warm,day_part
0,Panem - Carreta,2022-02-18,viernes,2022-02-18 08:21:04.887,2022-02-18 08:17:48.077,7,1,1,Para llevar,'-,...,NaN,NaN,NaN,NaN,NaN,DE005,detail_Panem-Carreta_2022-02-18_2022-05-07.csv,13.5,cold,morning
5,Panem - Carreta,2022-02-18,viernes,2022-02-18 08:29:26.893,2022-02-18 08:28:29.090,7,4,4,Para llevar,'-,...,NaN,NaN,NaN,NaN,NaN,PD015,detail_Panem-Carreta_2022-02-18_2022-05-07.csv,13.5,cold,morning
10,Panem - Carreta,2022-02-18,viernes,2022-02-18 08:37:58.597,2022-02-18 08:35:52.333,7,8,8,Para llevar,'-,...,NaN,NaN,NaN,NaN,NaN,RE007,detail_Panem-Carreta_2022-02-18_2022-05-07.csv,13.5,cold,morning
11,Panem - Carreta,2022-02-18,viernes,2022-02-18 08:37:58.597,2022-02-18 08:35:59.880,7,8,8,Para llevar,'-,...,NaN,NaN,NaN,NaN,NaN,RE004,detail_Panem-Carreta_2022-02-18_2022-05-07.csv,13.5,cold,morning
15,Panem - Carreta,2022-02-18,viernes,2022-02-18 08:45:46.553,2022-02-18 08:45:32.320,7,11,11,Para llevar,'-,...,NaN,NaN,NaN,NaN,NaN,PD015,detail_Panem-Carreta_2022-02-18_2022-05-07.csv,13.5,cold,morning
16,Panem - Carreta,2022-02-18,viernes,2022-02-18 08:46:51.277,2022-02-18 08:46:15.513,7,12,12,Para llevar,'-,...,NaN,NaN,NaN,NaN,NaN,PD009,detail_Panem-Carreta_2022-02-18_2022-05-07.csv,13.5,cold,morning
20,Panem - Carreta,2022-02-18,viernes,2022-02-18 08:58:18.827,2022-02-18 08:55:38.760,7,14,14,Para llevar,'-,...,NaN,NaN,NaN,NaN,NaN,PD006,detail_Panem-Carreta_2022-02-18_2022-05-07.csv,13.5,cold,morning
21,Panem - Carreta,2022-02-18,viernes,2022-02-18 08:58:18.827,2022-02-18 08:55:42.573,7,14,14,Para llevar,'-,...,NaN,NaN,NaN,NaN,NaN,PD007,detail_Panem-Carreta_2022-02-18_2022-05-07.csv,13.5,cold,morning
23,Panem - Carreta,2022-02-18,viernes,2022-02-18 09:00:22.110,2022-02-18 08:59:24.810,7,15,15,Para llevar,'-,...,NaN,NaN,NaN,NaN,NaN,PD001,detail_Panem-Carreta_2022-02-18_2022-05-07.csv,13.5,cold,morning
29,Panem - Carreta,2022-02-18,viernes,2022-02-18 09:09:34.970,2022-02-18 09:08:37.933,7,18,18,Para llevar,'-,...,NaN,NaN,NaN,NaN,NaN,RE004,detail_Panem-Carreta_2022-02-18_2022-05-07.csv,13.5,cold,morning


## Summary

After this notebook, `df` contains:
- ✅ Proper datetime types
- ✅ `is_modifier` as boolean
- ✅ Beverages removed
- ✅ Branch names standardized (7 unique branches)
- ✅ Product name typos fixed

**Next step:** `02_eda_sales.ipynb` — explore what sells, where, and how much.